In [0]:

from pyspark.sql.types import *
import pyspark.sql.functions F





def transform_sales(df):

    product_schema = StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", DoubleType()),
        StructField("qty", IntegerType()),
        StructField("unit", StringType()),
        
    ])

    sales_df = df.withColumn("product", F.from_json("product", product_schema )).select(
        "customer_id",
        "customer_name",
        "product_name", 
        "order_date",
        "product_category",
        "product.curr",
        "product.id",
        "product.name",
        "product.price",
        "product.qty",
        "product.unit",
        "total_price",
        "last_update_ts"
    )



df = spark.read.table("retaildataplatform.bronze.s3_sales")

flatten_df = transform_sales(df)



target_table = "retaildataplatform.silver.sales"

if spark.catalog.tableExists("retaildataplatform.silver.sales"):
    print("Table Exists - Now Proceeding with SCD 1")

    # Load Target

    target = DeltaTable.forName(spark, target_table)

    # SCD Type 1

    # Match in source and target -> Key exist in target
    (
        target
        .merge(
            cleaned_customer.alias("source"), 
            "target.customer_id = source.customer_id"

        )


    # Update - Match
    .whenMatchedUpdateAll()

    # Insert - Not Match
    .whenNotMatchedInsertAll()

    .execute()
    )


else:
    print("Table Does Not Exist - Now Creating Table")
    spark.sql("""Create Schema if not exists silver """)
    cleaned_customer.write.format("delta").mode("overwrite").saveAsTable("retaildataplatform.silver.customers")












In [0]:

from pyspark.sql.types import *
import pyspark.sql.functions as F





def transform_sales(df):

    product_schema = StructType([
        StructField("curr", StringType()),
        StructField("id", StringType()),
        StructField("name", StringType()),
        StructField("price", DoubleType()),
        StructField("qty", IntegerType()),
        StructField("unit", StringType()),
        
    ])

    sales_df = df.withColumn("product", F.from_json("product", product_schema )).select(
        "customer_id",
        "customer_name",
        "product_name", 
        "order_date",
        "product_category",
        "product.curr",
        "product.id",
        "product.name",
        "product.price",
        "product.qty",
        "product.unit",
        "total_price",
        "last_update_ts"
    )
    

    return sales_df



df = spark.read.table("retaildataplatform.bronze.s3_sales")

flatten_df = transform_sales(df)

In [0]:
flatten_df.display()